# 00_intro.ipynb

## Langchain Intro

uv add langchain langchain_openai langchain_core langchain_community

# 한줄 요약
Langchain == Agent Builder다.

In [1]:
# 환경변수(.env) 내용을 로드
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.agents import create_agent

# 메모리 추가
from langgraph.checkpoint.memory import InMemorySaver


# 매개변수 city에 들어갈 데이터 타입은 str이다.
def get_weather(city: str):
    '''주어진 도시의 날씨를 확인하는 Tool'''  # Tool Description (설명서)
    return f'{city}는 화창합니다.'


# 이름: calculator. 매개변수 num1, num2 (float), opr (str)
def calculator(num1: float, num2: float, opr: str):
    '''opr을 보고 num1 과 num2의 사칙연산을 진행
    opr 은 '+', '-'만 사용 가능
    '''
    if opr == '+':
        return num1 + num2
    elif opr == '-':
        return num1 - num2

memory = InMemorySaver()

agent = create_agent(
    model='openai:gpt-4.1-mini',
    tools=[get_weather, calculator],
    system_prompt="You are a helpful assistant. Answer in KOR",
    checkpointer=memory,  # 메모리
)

In [3]:
# 메모리에서 대화 내역을 구분할 key 가 필요
thread_config = {
    'configurable': {'thread_id': '1234'}  # 세션 번호
}

# 에이전트 실행
agent.invoke(
    # 1번인자: 메세지
    {
        'messages': [
            {'role': 'user', 'content': '아까 최종 계산 결과 얼마였지?'}
        ]
    },
    # 2번인자: 설정
    thread_config,
)

{'messages': [HumanMessage(content='아까 최종 계산 결과 얼마였지?', additional_kwargs={}, response_metadata={}, id='3c839157-104b-4400-8268-66085395f03b'),
  AIMessage(content='죄송하지만, 이전 대화에서 최종 계산 결과를 알려주셨거나 계산한 기록이 없어서 기억하고 있지 않습니다. 최종 계산 결과를 다시 말씀해 주시면 도와드리겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 118, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_bb832114e0', 'id': 'chatcmpl-EJDyIDM3GBJ8YWt50ItEIKRKbxbbj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a05c0b-6aa9-7ea3-b117-09d4d24f7628-0', tool_calls=[], invalid_too

In [ ]:
# 대화형 UI
user_input = input()

thread_config = {
    'configurable': {'thread_id': '4567'}  # 세션 번호
}

while user_input != '종료':
    print('사용자:', user_input)
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': user_input}]},
        thread_config,
    )
    print('AI:', result['messages'][-1].content)
    user_input = input()

# 실습

1. 로또 Agent 만들기

2. Tool
함수이름: get_lotto_info
requests 로 요청 보내는 도구
url : https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do?srchStrLtEpsd=1232&srchEndLtEpsd=1232
srchStrLtEpsd : 조회 시작 회차
srchEndLtEpsd : 조회 종료 회차
매개변수: start_round: int, end_round: int (각각 조회할 때 시작회차와 마지막 회차의 번호)
설명(description): LLM이 잘 사용할 수 있도록 작성

3. Memory
InMemorySaver
thread_id 는 편한대로

4. Model
openai:gpt-4.1-mini

5. System Prmopt
잘 동작하도록 작성
오늘 날짜 + 시간 넣기 (검색 필요)
오늘 날짜 기준 최신회차는 1239

6. 대화형 UI로 만들기

7. 예시 대화
Q: 가장 최근회차의 1등 당첨금액은 얼마야? -> A: 22.1억
Q: 저번 회차는 몇명이 1등 당첨됐어? -> A: 23명